# PyDI Data Integration Workflow: Movies

This notebook demonstrates comprehensive data integration using PyDI. We'll work with movie datasets to showcase the data integration pipeline from entity matching to data fusion.

## Table of Contents
  - [Datasets](#datasets)
- [Part 1: Data Loading and Profiling](#part-1-data-loading-and-profiling)
- [Part 2: Entity Matching](#part-2-entity-matching)
  - [Step 1: Blocking](#step-1-blocking)
  - [Step 2: Blocking Evaluation](#step-2-evaluate-blocking-against-ground-truth)
  - [Step 3: Entity Matching with Comparators](#step-3-entity-matching-with-comparators)
  - [Step 4: Entity Matching Evaluation](#step-4-evaluate-matching-against-ground-truth)
  - [Step 5: Machine Learning-based Matching Rules](#step-5-machine-learning-based-matching-rules)
- [Part 3: Data Fusion](#part-3-data-fusion)
  - [Step 1: Define Fusion Strategy](#step-1-define-fusion-strategy)
  - [Step 2: Run Fusion](#step-2-run-fusion)
  - [Step 3: Data Fusion Evaluation](#step-3-evaluate-data-fusion)

In [2]:
from pathlib import Path

# Paths relative to this notebook
NOTEBOOK_DIR = Path(".").resolve()
INPUT_DIR = NOTEBOOK_DIR / "input"
OUTPUT_DIR = NOTEBOOK_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Part 1: Data Loading and Profiling

In [3]:
from PyDI.io import load_xml

# Load Academy Awards dataset
academy_awards = load_xml(
    INPUT_DIR / "data" / "academy_awards.xml",
    name="academy_awards",
    nested_handling="aggregate"
)

# Load Actors dataset  
actors = load_xml(
    INPUT_DIR / "data" / "actors.xml",
    name="actors", 
    nested_handling="aggregate"
)

# Load Golden Globes dataset
golden_globes = load_xml(
    INPUT_DIR / "data" / "golden_globes.xml",
    name="golden_globes",
    nested_handling="aggregate"
)

# Display basic information
datasets = [academy_awards, actors, golden_globes]
names = ["Academy Awards", "Actors", "Golden Globes"]

total_records = sum(len(df) for df in datasets)
print(f"Total records across all datasets: {total_records:,}")

Total records across all datasets: 7,010


In [4]:
from PyDI.utils import DataProfiler

# Initialize the DataProfiler
profiler = DataProfiler()

for df, name in zip(datasets, names):
    profile = profiler.summary(df) # automatically prints some statistics and returns object containing stats

display(profile)

academy_awards:
  Rows: 4,580
  Columns: 6
  Total nulls: 11,028
  Null percentage: 40.1%
  Null counts per column:
    title: 12 (0.3%)
    actors_actor_name: 3,531 (77.1%)
    director_name: 4,172 (91.1%)
    oscar: 3,313 (72.3%)

actors:
  Rows: 151
  Columns: 6
  Total nulls: 0
  Null percentage: 0.0%

golden_globes:
  Rows: 2,279
  Columns: 6
  Total nulls: 3,677
  Null percentage: 26.9%
  Null counts per column:
    actors_actor_name: 54 (2.4%)
    director_name: 1,966 (86.3%)
    globe: 1,657 (72.7%)



{'rows': 2279,
 'columns': 6,
 'nulls_total': 3677,
 'nulls_per_column': {'id': 0,
  'title': 0,
  'actors_actor_name': 54,
  'date': 0,
  'director_name': 1966,
  'globe': 1657},
 'dtypes': {'id': 'object',
  'title': 'object',
  'actors_actor_name': 'object',
  'date': 'object',
  'director_name': 'object',
  'globe': 'object'}}

### Attribute Coverage Analysis

In [5]:
coverage = profiler.analyze_coverage(
    datasets=datasets,
    include_samples=True,
    sample_count=3  # Show 3 sample values per attribute
)

print("📊 Attribute coverage across datasets:")
display(coverage)

# Identify attributes suitable for entity matching
print("\n🔗 Attributes suitable for entity matching:")
matching_attrs = coverage[coverage['datasets_with_attribute'] >= 2]['attribute'].tolist()
print(f"Attributes available in 2+ datasets: {matching_attrs}")

📊 Attribute coverage across datasets:


,attribute,academy_awards_count,academy_awards_pct,academy_awards_coverage,academy_awards_samples,actors_count,actors_pct,actors_coverage,actors_samples,golden_globes_count,golden_globes_pct,golden_globes_coverage,golden_globes_samples,avg_coverage,max_coverage,datasets_with_attribute
0,actors_actor_birthday,0/0,0%,0.000000,N/A,151/151,100.0%,1.0,"['1906-01-01', '1892-01-01', '1902-01-01']",0/0,0%,0.000000,N/A,0.333333,1.000000,1
1,actors_actor_birthplace,0/0,0%,0.000000,N/A,151/151,100.0%,1.0,"['Pennsylvania', 'Canada', 'Canada']",0/0,0%,0.000000,N/A,0.333333,1.000000,1
2,actors_actor_name,1049/4580,22.9%,0.229039,"['Javier Bardem', ['Jeff Bridges', 'Hailee Ste...",151/151,100.0%,1.0,"['Janet Gaynor', 'Mary Pickford', 'Norma Shear...",2225/2279,97.6%,0.976305,"['Halle Berry', 'Nicole Kidman', 'Jennifer Law...",0.735115,1.000000,3
3,date,4580/4580,100.0%,1.000000,"['2010-01-01', '2010-01-01', '2010-01-01']",151/151,100.0%,1.0,"['1929-01-01', '1930-01-01', '1931-01-01']",2279/2279,100.0%,1.000000,"['2011-01-01', '2011-01-01', '2011-01-01']",1.000000,1.000000,3
4,director_name,408/4580,8.9%,0.089083,"['Joel Coen and Ethan Coen', 'David Fincher', ...",0/0,0%,0.0,N/A,313/2279,13.7%,0.137341,"['Darren Aronofsky', 'David Fincher', 'Tom Hoo...",0.075475,0.137341,2
5,globe,0/0,0%,0.000000,N/A,0/0,0%,0.0,N/A,622/2279,27.3%,0.272927,"['yes', 'yes', 'yes']",0.090976,0.272927,1
6,id,4580/4580,100.0%,1.000000,"['academy_awards_1', 'academy_awards_2', 'acad...",151/151,100.0%,1.0,"['actors_1', 'actors_2', 'actors_3']",2279/2279,100.0%,1.000000,"['golden_globes_1', 'golden_globes_2', 'golden...",1.000000,1.000000,3
7,oscar,1267/4580,27.7%,0.276638,"['yes', 'yes', 'yes']",0/0,0%,0.0,N/A,0/0,0%,0.000000,N/A,0.092213,0.276638,1
8,title,4568/4580,99.7%,0.997380,"['Biutiful', 'True Grit', 'The Social Network']",151/151,100.0%,1.0,"['7th Heaven', 'Coquette', 'The Divorcee']",2279/2279,100.0%,1.000000,"['Frankie and Alice', 'Rabbit Hole', ""Winter's...",0.999127,1.000000,3



🔗 Attributes suitable for entity matching:
Attributes available in 2+ datasets: ['actors_actor_name', 'date', 'director_name', 'id', 'title']


### Detailed Data Profiling

In [6]:
from pathlib import Path

# Generate detailed HTML profiles for each dataset
profile_dir = OUTPUT_DIR / "dataset-profiles"
profile_dir.mkdir(parents=True, exist_ok=True)

profile_paths = []

for df, name in zip(datasets, names):
    print(f"Profiling {name}...")
    
    profile_path = profiler.profile(df, str(profile_dir))
    profile_paths.append(profile_path)
    print(f"Profile saved: {profile_path}")

print(f"\n Generated {len(profile_paths)} detailed HTML reports")
print(f" Location: {profile_dir}")
print("\n Open these HTML files in your browser for interactive exploration:")
for path in profile_paths:
    print(f"  • {Path(path).name}")


Profiling Academy Awards...


/usr/local/Caskroom/miniconda/base/envs/pydi/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 517.88it/s]


Profile saved: /Users/luca/PycharmProjects/PyDI/usecases/movies/output/dataset-profiles/academy_awards_profile.html
Profiling Actors...


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 23.88it/s]


Profile saved: /Users/luca/PycharmProjects/PyDI/usecases/movies/output/dataset-profiles/actors_profile.html
Profiling Golden Globes...


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 354.34it/s]

Profile saved: /Users/luca/PycharmProjects/PyDI/usecases/movies/output/dataset-profiles/golden_globes_profile.html

 Generated 3 detailed HTML reports
 Location: /Users/luca/PycharmProjects/PyDI/usecases/movies/output/dataset-profiles

 Open these HTML files in your browser for interactive exploration:
  • academy_awards_profile.html
  • actors_profile.html
  • golden_globes_profile.html


## Part 2: Entity Matching

### Step 1: Blocking

In [7]:
# Set up logging
import logging

import os
os.makedirs('output/logs', exist_ok=True)

logging.basicConfig(
    level=logging.INFO, # Alternatively, use logging.DEBUG for more verbosity
    format='[%(levelname)-5s] %(name)s - %(message)s',
    handlers=[
          logging.FileHandler('output/logs/pydi.log'),  # Save to file
          logging.StreamHandler()                      # Display on console
      ],
    force=True
)

In [8]:
# Import blocking methods
from PyDI.entitymatching import SortedNeighbourhoodBlocker, TokenBlocker

token_blocker_a2g = TokenBlocker(
    actors, golden_globes,
    column='title',      # Tokenize titles
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id',
)

sn_blocker_aa2a = SortedNeighbourhoodBlocker(
    academy_awards, actors,
    key='title',  # Sort by title
    window=30,     # Compare with 30 neighbors
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)

[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - created 288 token keys for first dataset
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - created 2384 token keys for second dataset
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - created 245 blocks from token keys
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - Debug results written to file: /Users/luca/PycharmProjects/PyDI/usecases/movies/output/blocking-evaluation/debugResultsBlocking_TokenBlocker.csv
[INFO ] PyDI.entitymatching.blocking.sorted_neighbourhood.SortedNeighbourhoodBlocker - created sorted neighbourhood with window size 30
[INFO ] PyDI.entitymatching.blocking.sorted_neighbourhood.SortedNeighbourhoodBlocker - created 1 sorted sequence from 4731 records
[INFO ] PyDI.entitymatching.blocking.sorted_neighbourhood.SortedNeighbourhoodBlocker - Debug results written to file: /Users/luca/PycharmProjects/PyDI/usecases/movies/output/blocking-evaluation/debugRe

### Step 2: Evaluate Blocking Against Ground Truth

In [9]:
import pandas as pd
from PyDI.io import load_csv
from PyDI.entitymatching import EntityMatchingEvaluator
# Showcase EntityMatchingEvaluator.evaluate_blocking utility

# Load test set with proper column names
test_gt = load_csv(
    INPUT_DIR / "entitymatching" / "actors_2_golden_globes_test.csv",
    name="test_set", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Use EntityMatchingEvaluator.evaluate_blocking on Standard Blocking
results = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=token_blocker_a2g,
    test_pairs=test_gt,
    out_dir=OUTPUT_DIR / "blocking-evaluation"
)

display(results)

[INFO ] root - Starting batched blocking evaluation...
[INFO ] root - Processed 10 batches, 10000 pairs, 6 true matches
[INFO ] root - Processed 20 batches, 20000 pairs, 8 true matches
[INFO ] root - Processed 30 batches, 30000 pairs, 14 true matches
[INFO ] root -   Pair Completeness: 0.962
[INFO ] root -   Pair Quality:      0.001
[INFO ] root -   Reduction Ratio:   0.902688
[INFO ] root -   True Matches Found: 25/26
[INFO ] root -   Batches Processed:  34
[INFO ] root - Blocking evaluation complete!


{'pair_completeness': 0.9615384615384616,
 'pair_quality': 0.0007465360726230291,
 'reduction_ratio': 0.9026876549201026,
 'total_candidates': 33488,
 'total_possible_pairs': 344129,
 'true_positives_found': 25,
 'total_true_pairs': 26,
 'batches_processed': 34,
 'evaluation_timestamp': '2026-01-16T15:58:29.816115',
 'output_files': ['/Users/luca/PycharmProjects/PyDI/usecases/movies/output/blocking-evaluation/blocking_evaluation_summary.json',
  '/Users/luca/PycharmProjects/PyDI/usecases/movies/output/blocking-evaluation/blocking_detailed_results.csv']}

Now let's evaluate which blocking method we want to use for each dataset combination:

In [10]:
# Load test set with proper column names
test_gt = load_csv(
    INPUT_DIR / "entitymatching" / "academy_awards_2_actors_test.csv",
    name="test_set", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Use EntityMatchingEvaluator.evaluate_blocking on Standard Blocking
results = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=sn_blocker_aa2a,
    test_pairs=test_gt,
    out_dir=OUTPUT_DIR / "blocking-evaluation"
)

display(results)

[INFO ] root - Starting batched blocking evaluation...
[INFO ] root -   Pair Completeness: 0.979
[INFO ] root -   Pair Quality:      0.005
[INFO ] root -   Reduction Ratio:   0.987244
[INFO ] root -   True Matches Found: 46/47
[INFO ] root -   Batches Processed:  9
[INFO ] root - Blocking evaluation complete!


{'pair_completeness': 0.9787234042553191,
 'pair_quality': 0.0052142371344366355,
 'reduction_ratio': 0.9872437028254143,
 'total_candidates': 8822,
 'total_possible_pairs': 691580,
 'true_positives_found': 46,
 'total_true_pairs': 47,
 'batches_processed': 9,
 'evaluation_timestamp': '2026-01-16T15:58:32.438384',
 'output_files': ['/Users/luca/PycharmProjects/PyDI/usecases/movies/output/blocking-evaluation/blocking_evaluation_summary.json',
  '/Users/luca/PycharmProjects/PyDI/usecases/movies/output/blocking-evaluation/blocking_detailed_results.csv']}

### Step 3: Entity Matching with Comparators

In [11]:
from PyDI.entitymatching import StringComparator, DateComparator, NumericComparator

# Create comparators for different attributes
comparators = [
    # Title similarity - most important for movies
    StringComparator(
        column='title',
        similarity_function='levenshtein', 
        preprocess=str.lower
    ),
    
    # Date proximity - movies from same year likely same film
    DateComparator(
        column='date', 
        #max_days_difference=365 
    ),
    
    # Actor name similarity - supporting evidence
    StringComparator(
        column='actors_actor_name',
        similarity_function='jaccard',
        preprocess=str.lower,
        list_strategy='concatenate'
    )
]

Next, we setup the matcher and run the matching with our chosen best blocking method:

In [12]:
from PyDI.entitymatching import RuleBasedMatcher

# Initialize Rule-Based Matcher
matcher = RuleBasedMatcher()

correspondences_a2g = matcher.match(
    df_left=actors,
    df_right=golden_globes, 
    candidates=token_blocker_a2g, # pass the blocker, which will internally generate candidate pairs using batching
    comparators=comparators,
    weights=[0.6, 0.3, 0.1],  # Title most important, then date, then actor,
    threshold=0.6, # set a similarity threshold for a match
    id_column='id'
)

[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 151 x 2279 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 151 x 2279 elements after 0:00:0.034; 33488 blocked pairs (reduction ratio: 0.9026876549201026)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:34.265; found 106 correspondences.


In [13]:
matcher = RuleBasedMatcher()

correspondences_aa2a = matcher.match(
    df_left=academy_awards,
    df_right=actors, 
    candidates=sn_blocker_aa2a,
    comparators=comparators,
    weights=[0.6, 0.3, 0.1],  # Title most important, then date, then actor,
    threshold=0.6,
    id_column='id'
)

[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 4580 x 151 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 4580 x 151 elements after 0:00:0.019; 8822 blocked pairs (reduction ratio: 0.9872437028254143)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:8.533; found 160 correspondences.


### Step 4: Evaluate Matching Against Ground Truth

In [14]:
gt_test = load_csv(
    INPUT_DIR / "entitymatching" / "actors_2_golden_globes_test.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_a2g,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

display(eval_results)

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  23
[INFO ] root -   True Negatives:  54
[INFO ] root -   False Positives: 2
[INFO ] root -   False Negatives: 3
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.939
[INFO ] root -   Precision: 0.920
[INFO ] root -   Recall:    0.885
[INFO ] root -   F1-Score:  0.902


{'precision': 0.92,
 'recall': 0.8846153846153846,
 'f1': 0.9019607843137256,
 'accuracy': 0.9390243902439024,
 'true_positives': 23,
 'false_positives': 2,
 'false_negatives': 3,
 'true_negatives': 54,
 'threshold_used': 0.0,
 'total_correspondences': 106,
 'filtered_correspondences': 106,
 'evaluation_timestamp': '2026-01-16T15:59:15.741695',
 'output_files': ['/Users/luca/PycharmProjects/PyDI/usecases/movies/output/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/luca/PycharmProjects/PyDI/usecases/movies/output/debug_results_entity_matching/matching_detailed_results.csv']}

In [15]:
print("Analyzing cluster size distribution in our entity matching results...")

# Create cluster size distribution from our matches
cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_a2g,
    out_dir=str(OUTPUT_DIR / "cluster_analysis")
)

print(f"\n📊 Cluster Size Distribution Results:")
display(cluster_distribution)

[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 98 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	94	|	95.92%
[INFO ] PyDI.entitymatching.evaluation - 		3	|	3	|	3.06%
[INFO ] PyDI.entitymatching.evaluation - 		7	|	1	|	1.02%
[INFO ] root - Cluster size distribution written to /Users/luca/PycharmProjects/PyDI/usecases/movies/output/cluster_analysis/cluster_size_distribution.csv


Analyzing cluster size distribution in our entity matching results...

📊 Cluster Size Distribution Results:


,cluster_size,frequency,percentage
0,2,94,95.918367
1,3,3,3.061224
2,7,1,1.020408


Additionally, PyDI offers 6 different post-clustering methods to "clean" clusters after entity matching. For example, if we want to enforce that each record in a dataset can only have exactly one correspondence in the other dataset, we can apply a greedy one-to-one matching, maximum bipartite matching or stable marriage matching.

In [16]:
from PyDI.entitymatching import MaximumBipartiteMatching, StableMatching

# use Maximum Bipartite Matching to refine results to 1:1 matches
clusterer = MaximumBipartiteMatching()
correspondences_a2g = clusterer.cluster(correspondences_a2g)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_a2g,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

display(eval_results)

[INFO ] root - Filtered correspondences: 106 -> 106 (threshold=0.0)
[INFO ] root - Maximum bipartite matching: 106 -> 98 
[INFO ] root - MaximumBipartiteMatching: 106 -> 98 correspondences
[INFO ] root - MaximumBipartiteMatching: 204 -> 196 entities
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  22
[INFO ] root -   True Negatives:  55
[INFO ] root -   False Positives: 1
[INFO ] root -   False Negatives: 4
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.939
[INFO ] root -   Precision: 0.957
[INFO ] root -   Recall:    0.846
[INFO ] root -   F1-Score:  0.898


{'precision': 0.9565217391304348,
 'recall': 0.8461538461538461,
 'f1': 0.8979591836734695,
 'accuracy': 0.9390243902439024,
 'true_positives': 22,
 'false_positives': 1,
 'false_negatives': 4,
 'true_negatives': 55,
 'threshold_used': 0.0,
 'total_correspondences': 98,
 'filtered_correspondences': 98,
 'evaluation_timestamp': '2026-01-16T15:59:15.850222',
 'output_files': ['/Users/luca/PycharmProjects/PyDI/usecases/movies/output/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/luca/PycharmProjects/PyDI/usecases/movies/output/debug_results_entity_matching/matching_detailed_results.csv']}

In [17]:
from PyDI.entitymatching import  GreedyOneToOneMatchingAlgorithm

gt_test = load_csv(
    INPUT_DIR / "entitymatching" / "academy_awards_2_actors_test.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_aa2a,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_aa2a,
)

clusterer = GreedyOneToOneMatchingAlgorithm()
correspondences_aa2a = clusterer.cluster(correspondences_aa2a)


cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_aa2a,
)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_aa2a,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  44
[INFO ] root -   True Negatives:  3294
[INFO ] root -   False Positives: 6
[INFO ] root -   False Negatives: 3
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.997
[INFO ] root -   Precision: 0.880
[INFO ] root -   Recall:    0.936
[INFO ] root -   F1-Score:  0.907
[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 142 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	128	|	90.14%
[INFO ] PyDI.entitymatching.evaluation - 		3	|	12	|	8.45%
[INFO ] PyDI.entitymatching.evaluation - 		4	|	1	|	0.70%
[INFO ] PyDI.entitymatching.evaluation - 		6	|	1	|	0.70%
[INFO ] root - Filtered correspondences: 160 -> 160 (threshold=0.0)
[INFO ] root - Greedy matching: 160 -> 143 correspondences (286 entities matched)
[INFO ] 

## Part 3: Data Fusion

In [18]:
academy_awards["academy_awards_id"] = academy_awards["id"]

# Assign trust scores to datasets
academy_awards.attrs["trust_score"] = 3
actors.attrs["trust_score"] = 2
golden_globes.attrs["trust_score"] = 1

all_correspondences = pd.concat([correspondences_a2g, correspondences_aa2a], ignore_index=True)
print(f'Total correspondences: {len(all_correspondences):,}')

Total correspondences: 241


## Step 1: Define Fusion Strategy 

In [19]:
from PyDI.fusion import DataFusionStrategy, longest_string, union, prefer_higher_trust

strategy = DataFusionStrategy('movie_fusion_strategy')

strategy.add_attribute_fuser('title', longest_string)
strategy.add_attribute_fuser('director_name', longest_string)
strategy.add_attribute_fuser('date', prefer_higher_trust, trust_key="trust_score")

strategy.add_attribute_fuser('actors_actor_name', union)

[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'title' using rule 'longest_string'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'director_name' using rule 'longest_string'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'date' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'actors_actor_name' using rule 'union'


## Step 2: Run Fusion

In [20]:
from PyDI.fusion import DataFusionEngine

engine = DataFusionEngine(strategy, debug=True, debug_format='json',debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion.jsonl")

fused = engine.run(
    datasets=[academy_awards, actors, golden_globes],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False,
)
print(f'Fused rows: {len(fused):,}')
display(fused.head(5))

[INFO ] PyDI.fusion.engine - Fusion debug logging enabled; refer to /Users/luca/PycharmProjects/PyDI/usecases/movies/output/data_fusion/debug_fusion.jsonl for detailed traces.
[INFO ] PyDI.fusion.engine - Starting data fusion with strategy 'movie_fusion_strategy'
[INFO ] PyDI.fusion.engine - *    Loading correspondences    *
[INFO ] PyDI.fusion.engine - Correspondence ID coverage: matched 390 of 390 unique IDs
[INFO ] PyDI.fusion.engine - Created 6769 record groups from 241 correspondences
[INFO ] PyDI.fusion.engine - Group Size Distribution of 6769 clusters:
[INFO ] PyDI.fusion.engine - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.fusion.engine - 	──────────────────────────────────────────────────
[INFO ] PyDI.fusion.engine - 		2	|	57	|	0.84%
[INFO ] PyDI.fusion.engine - 		3	|	92	|	1.36%
[INFO ] PyDI.fusion.engine - Attribute Consistencies:
[INFO ] PyDI.fusion.engine -     _id: 0.00
[INFO ] PyDI.fusion.engine -     academy_awards_id: 1.00
[INFO ] PyDI.fusion.engine -     actors

Fused rows: 149


,_id,_fusion_sources,_fusion_source_datasets,academy_awards_id,actors_actor_birthday,actors_actor_birthplace,actors_actor_name,date,director_name,globe,id,oscar,title,_fusion_confidence,_fusion_metadata
0,academy_awards_345,"[academy_awards_345, actors_78, golden_globes_...","[academy_awards, actors, golden_globes]",academy_awards_345,1974-01-01,Washington,"[Clint Eastwood, Hilary Swank, Morgan Freeman]",2004-01-01,Clint Eastwood,yes,academy_awards_345,yes,Million Dollar Baby,0.611111,"{'_id_rule': 'first_non_null', '_id_inputs': [..."
1,academy_awards_346,"[academy_awards_346, actors_151, golden_globes...","[academy_awards, actors, golden_globes]",academy_awards_346,1967-01-01,Texas,[Jamie Foxx],2004-01-01,Taylor Hackford,yes,academy_awards_346,yes,Ray,0.666667,"{'_id_rule': 'first_non_null', '_id_inputs': [..."
2,academy_awards_402,"[academy_awards_402, actors_150, golden_globes...","[academy_awards, actors, golden_globes]",academy_awards_402,1960-01-01,California,"[Marcia Gay Harden, Sean Penn, Tim Robbins]",2003-01-01,Clint Eastwood,yes,academy_awards_402,yes,Mystic River,0.611111,"{'_id_rule': 'first_non_null', '_id_inputs': [..."
3,academy_awards_409,"[academy_awards_409, actors_77, golden_globes_...","[academy_awards, actors, golden_globes]",academy_awards_409,1975-01-01,South Africa,[Charlize Theron],2003-01-01,None,yes,academy_awards_409,yes,Monster,0.555556,"{'_id_rule': 'first_non_null', '_id_inputs': [..."
4,academy_awards_448,"[academy_awards_448, actors_149]","[academy_awards, actors]",academy_awards_448,1973-01-01,New York,[Adrien Brody],2002-01-01,Roman Polanski,NaN,academy_awards_448,yes,The Pianist,0.687500,"{'_id_rule': 'first_non_null', '_id_inputs': [..."


## Step 3: Evaluate Data Fusion

In [21]:
from PyDI.fusion import tokenized_match, year_only_match, boolean_match

strategy.add_evaluation_function("title", tokenized_match)
strategy.add_evaluation_function("director_name", tokenized_match)
strategy.add_evaluation_function("actors_actor_name", tokenized_match)
strategy.add_evaluation_function("date", year_only_match)
strategy.add_evaluation_function("oscar", boolean_match)

[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'title'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'director_name'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'actors_actor_name'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'date'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'oscar'


In [ ]:
from PyDI.fusion import DataFusionEvaluator

fusion_test_set = load_xml(INPUT_DIR / 'fusion' / 'test_set.xml', name='fusion_test_set', nested_handling='aggregate')

# Create evaluator with our fusion strategy
evaluator = DataFusionEvaluator(strategy, debug=True, debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion_eval.jsonl", debug_format="json")

# Evaluate the fused results against the gold standard
print("Evaluating fusion results against gold standard...")
evaluation_results = evaluator.evaluate(
    fused_df=fused,
    fused_id_column='academy_awards_id',
    gold_df=fusion_test_set,
    gold_id_column='id',
)

# Display evaluation metrics
print("\nFusion Evaluation Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.3f}")
    else:
        print(f"  {metric}: {value}")
        
print(f"\nOverall Accuracy: {evaluation_results.get('overall_accuracy', 0):.1%}")

[INFO ] PyDI.fusion.evaluation - Fusion evaluation debug logging enabled; refer to /Users/luca/PycharmProjects/PyDI/usecases/movies/output/data_fusion/debug_fusion_eval.jsonl for mismatch details.
[INFO ] PyDI.fusion.evaluation - Starting fusion evaluation
[INFO ] PyDI.fusion.evaluation - Evaluation complete: 1.000 overall accuracy (120/120)
[INFO ] PyDI.fusion.evaluation - Evaluation mismatches by attribute (debug): none recorded


Evaluating fusion results against gold standard...

Fusion Evaluation Results:
  overall_accuracy: 1.000
  macro_accuracy: 1.000
  num_evaluated_records: 24
  num_evaluated_attributes: 5
  total_evaluations: 120
  total_correct: 120
  title_accuracy: 1.000
  title_count: 24
  actors_actor_name_accuracy: 1.000
  actors_actor_name_count: 24
  oscar_accuracy: 1.000
  oscar_count: 24
  director_name_accuracy: 1.000
  director_name_count: 24
  date_accuracy: 1.000
  date_count: 24

Overall Accuracy: 100.0%
